# Traffic intersection simulation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plugboard-dev/plugboard/blob/main/examples/demos/transport/001_traffic_simulation/traffic-simulation.ipynb)

[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/plugboard-dev/plugboard)

<div class="alert alert-block alert-warning">
Event-driven simulations are non-deterministic and not suitable for running in distributed environments, because Plugboard won't guarantee the order in which events get processed.
</div>


This demo simulates a dual-intersection traffic corridor using Plugboard's **event-driven** architecture.
Vehicles arrive at **Junction A**, wait for a green signal, cross the intersection, travel along a
connecting road with stochastic travel time, then repeat the process at **Junction B** before exiting the system.

### Components

| Component | Role |
|---|---|
| **SimulationClock** | Emits integer time steps (1 step = 1 second) |
| **VehicleSource** | Generates vehicles via a Poisson process |
| **TrafficLight** ×2 | FSM controllers cycling GREEN → YELLOW → RED |
| **Junction** ×2 | FIFO queues that release vehicles when the signal is green |
| **ConnectingRoad** | Adds triangular-distributed travel time between junctions |
| **MetricsCollector** | Subscribes to *all* event types for post-simulation analysis |

### Event types

| Event | Trigger |
|---|---|
| `VehicleArrivedEvent` | A vehicle enters a junction queue |
| `VehicleDepartedEvent` | A vehicle passes through a junction |
| `SignalChangedEvent` | A traffic light transitions between states |
| `CongestionAlertEvent` | A junction's queue exceeds its congestion threshold |

Regular connectors carry continuous data (clock time, signal state, queue length), while
**events** capture discrete occurrences — the two paradigms work together in a single process.

We first run a **synchronised** baseline (both lights share the same phase), then show how
offsetting Junction B's green phase to create a **green wave** improves throughput.

In [ ]:
# Install plugboard and dependencies for Google Colab
!pip install -q plugboard plotly

In [ ]:
import random

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from plugboard.connector import AsyncioConnector
from plugboard.process import LocalProcess
from plugboard.schemas import ConnectorSpec

from traffic_components import (
    ConnectingRoad,
    Junction,
    MetricsCollector,
    SimulationClock,
    TrafficLight,
    VehicleSource,
)

connect = lambda src, tgt: AsyncioConnector(spec=ConnectorSpec(source=src, target=tgt))

## Assembling the process

We create 8 components and wire them together:

- **Field connectors** carry the clock time to every component, signal states from lights to junctions,
  and queue metrics into the collector.
- **Event connectors** are built automatically from the `input_events` / `output_events` declared on each
  component's `IOController`. Plugboard routes every emitted event to all components that subscribe to its type.

The baseline scenario uses synchronised signals — both junctions share the same GREEN/YELLOW/RED phase.

In [ ]:
random.seed(42)  # reproducible results

DURATION = 600        # seconds (10 minutes)
ARRIVAL_RATE = 0.4    # vehicles per second (~24 vehicles/min)
GREEN = 30            # seconds
YELLOW = 5
RED = 30

components = [
    SimulationClock(name="clock", duration=DURATION),
    VehicleSource(name="source", arrival_rate=ARRIVAL_RATE, target_junction="A"),
    TrafficLight(name="light-a", junction="A", green_duration=GREEN, yellow_duration=YELLOW, red_duration=RED),
    TrafficLight(name="light-b", junction="B", green_duration=GREEN, yellow_duration=YELLOW, red_duration=RED),
    Junction(name="junction-a", junction_name="A", congestion_threshold=15),
    Junction(name="junction-b", junction_name="B", congestion_threshold=15),
    ConnectingRoad(name="road-ab", from_junction="A", to_junction="B"),
    MetricsCollector(name="metrics"),
]

connectors = [
    # Clock drives every component
    connect("clock.time", "source.time"),
    connect("clock.time", "light-a.time"),
    connect("clock.time", "light-b.time"),
    connect("clock.time", "junction-a.time"),
    connect("clock.time", "junction-b.time"),
    connect("clock.time", "road-ab.time"),
    connect("clock.time", "metrics.time"),
    # Signal state into junctions
    connect("light-a.signal", "junction-a.signal"),
    connect("light-b.signal", "junction-b.signal"),
    # Queue metrics into collector
    connect("junction-a.queue_length", "metrics.queue_a"),
    connect("junction-b.queue_length", "metrics.queue_b"),
    connect("junction-a.total_passed", "metrics.passed_a"),
    connect("junction-b.total_passed", "metrics.passed_b"),
]

# Event connectors are built automatically from IO declarations
event_connectors = AsyncioConnector.builder().build_event_connectors(components)

process = LocalProcess(components=components, connectors=connectors + event_connectors)
print(f"{len(process.components)} components, {len(process.connectors)} connectors")

In [ ]:
from plugboard.diagram import MermaidDiagram

diagram_url = MermaidDiagram.from_process(process).url
print(diagram_url)

In the process diagram dotted lines represent the event-driven portions of the model.

![Process Diagram](https://mermaid.ink/img/pako:eNrtmF1v2yAUhv-KxWWUL3_EX5qqVWlvpvammXYxRYqwwTYbhgjjrF2U_z4ISbtOS6cGcre7QN7z8vhw8LG9BSVHGOSgovxH2UAhvbuHJfO8kvLy-8et1zVwjXNP8J4hjIYehQWmubcgbU-hJJzNtfBDIa4Gg33MYODtvNHoyut4L0p82uILbkhJ8WIvMwYmRDu4QaCkbuQInvb4LGBVkfJO64zBIcQ1Q_F-hsIdw7eelVrwVio-HTQm_iXiAhTFuykc5kJwiEbwDYQ5ZwyrZVn9oKTG4hDkjqLFUpCyO-1xbwRzTqmC4cKYHMKOHHb17aw07ErcWW3YXorDjbG9nEug_M_Kbygb03xWUAiywejFrcGPf3aoa6O53WAmNcTY0bFxxmBxZo4MCK9V7_8HxM1B9IrCyf2Usxp3mmwFKRby7xTzZ9W1Fr2icFMVHakZpCv1HKRWOpGLxV4zN5ILMNhWhVuK8-vC0Y7YPEQakPMT6qLFaoJzy8pFb7Vd3759aAKbA-6Owqao3fQvp7mworDNhe1t3_ZsgiFosWghQerldauJlkA2uMVLkKufCFewp3IJlkxLYS_54omVIJeix0OgYOsG5BWknRr1awQlviGwFrA9SszkLSLq1nSco4ofq9EWyKe1fmuuSSeVvdrSitR6vhdUTTdSrrt8MtF_j2sim74Yl7yddATpV-xmk8WTOIhTGIQ4TkI4C0NUFn6WVkHkVyiZ-gEEu90QrCH7yvkzkxrqRR5B7vvxOE3iKJn5WTabRkE6BE8gT7JxlKZJEiSzMAijKFIeP_cG03Hip2E2TeIgnc6yMFUBeH9x9-YDwP47wBDUQmf0kCWsdlTM1cZKkM92vwAU_pKm)

## Run the baseline simulation

With both signals synchronised, vehicles released from Junction A arrive at Junction B
during its RED phase — they queue up and wait.

In [ ]:
async with process:
    await process.run()

In [ ]:
mc = process.components["metrics"]

df_queue = pd.DataFrame(mc.queue_history)
df_arrivals = pd.DataFrame(mc.arrival_log)
df_departures = pd.DataFrame(mc.departure_log)
df_signals = pd.DataFrame(mc.signal_log)
df_congestion = pd.DataFrame(mc.congestion_log)

total_arrived = len(df_arrivals[df_arrivals["target_junction"] == "A"])
passed_a = df_queue["passed_a"].iloc[-1]
passed_b = df_queue["passed_b"].iloc[-1]

print(f"Vehicles generated : {total_arrived}")
print(f"Through Junction A : {passed_a}")
print(f"Through Junction B : {passed_b}  (exited system)")
print(f"System throughput  : {passed_b / total_arrived:.1%}")
print(f"Congestion alerts  : {len(df_congestion)}")
if len(df_departures):
    print(f"Mean wait time (A) : {df_departures[df_departures['from_junction']=='A']['wait_time'].mean():.1f}s")
    print(f"Mean wait time (B) : {df_departures[df_departures['from_junction']=='B']['wait_time'].mean():.1f}s")

### Queue dynamics

The chart below shows how queue lengths at each junction oscillate with the signal cycle.
Green and red background bands indicate the signal phase at Junction A.

In [ ]:
CYCLE = GREEN + YELLOW + RED

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Junction A queue", "Junction B queue"))

# Signal-state bands for Junction A
for start in range(0, DURATION, CYCLE):
    fig.add_vrect(x0=start, x1=start + GREEN, fillcolor="green", opacity=0.08, line_width=0, row=1, col=1)
    fig.add_vrect(x0=start + GREEN + YELLOW, x1=start + CYCLE, fillcolor="red", opacity=0.08, line_width=0, row=1, col=1)

# Signal-state bands for Junction B (no offset in baseline)
for start in range(0, DURATION, CYCLE):
    fig.add_vrect(x0=start, x1=start + GREEN, fillcolor="green", opacity=0.08, line_width=0, row=2, col=1)
    fig.add_vrect(x0=start + GREEN + YELLOW, x1=start + CYCLE, fillcolor="red", opacity=0.08, line_width=0, row=2, col=1)

fig.add_trace(go.Scatter(x=df_queue["time"], y=df_queue["queue_a"], name="Queue A",
                         line=dict(color="royalblue")), row=1, col=1)
fig.add_trace(go.Scatter(x=df_queue["time"], y=df_queue["queue_b"], name="Queue B",
                         line=dict(color="coral")), row=2, col=1)

# Mark congestion alerts
if len(df_congestion):
    for _, row in df_congestion.iterrows():
        r = 1 if row["junction"] == "A" else 2
        fig.add_vline(x=row["time"], line_dash="dot", line_color="red", opacity=0.6, row=r, col=1)

fig.update_layout(height=500, template="plotly_white", showlegend=False,
                  title_text="Baseline: synchronised signals")
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="Vehicles", row=1, col=1)
fig.update_yaxes(title_text="Vehicles", row=2, col=1)
fig.show()

### Wait-time distributions

Vehicles arriving during a red phase wait longer. The distribution at Junction B is typically
wider because it includes both signal delays and the connecting-road variance.

In [ ]:
fig = go.Figure()
for junc, colour in [("A", "royalblue"), ("B", "coral")]:
    waits = df_departures[df_departures["from_junction"] == junc]["wait_time"]
    if len(waits):
        fig.add_trace(go.Histogram(x=waits, name=f"Junction {junc}", opacity=0.7,
                                   marker_color=colour, nbinsx=30))

fig.update_layout(barmode="overlay", template="plotly_white",
                  title="Vehicle wait-time distribution",
                  xaxis_title="Wait time (s)", yaxis_title="Count")
fig.show()

## Experiment: green-wave offset

When both signals share the same phase, vehicles released from Junction A during its green
phase arrive at Junction B roughly 10–25 seconds later — often hitting B's red phase.

A **green wave** staggers Junction B's phase so that its green window aligns with the arrival
of vehicles from A. The optimal offset is approximately:

$$\text{offset}_B = \text{cycle length} - \text{avg. travel time} \approx 65 - 15 = 50\text{s}$$

Let's re-run the simulation with this offset and compare.

In [ ]:
def build_process(light_b_offset: int = 0, seed: int = 42) -> LocalProcess:
    """Create a fresh traffic-corridor process with a given offset for Junction B."""
    random.seed(seed)
    comps = [
        SimulationClock(name="clock", duration=DURATION),
        VehicleSource(name="source", arrival_rate=ARRIVAL_RATE, target_junction="A"),
        TrafficLight(name="light-a", junction="A", green_duration=GREEN, yellow_duration=YELLOW, red_duration=RED),
        TrafficLight(name="light-b", junction="B", green_duration=GREEN, yellow_duration=YELLOW, red_duration=RED, offset=light_b_offset),
        Junction(name="junction-a", junction_name="A", congestion_threshold=15),
        Junction(name="junction-b", junction_name="B", congestion_threshold=15),
        ConnectingRoad(name="road-ab", from_junction="A", to_junction="B"),
        MetricsCollector(name="metrics"),
    ]
    conns = [
        connect("clock.time", "source.time"),
        connect("clock.time", "light-a.time"),
        connect("clock.time", "light-b.time"),
        connect("clock.time", "junction-a.time"),
        connect("clock.time", "junction-b.time"),
        connect("clock.time", "road-ab.time"),
        connect("clock.time", "metrics.time"),
        connect("light-a.signal", "junction-a.signal"),
        connect("light-b.signal", "junction-b.signal"),
        connect("junction-a.queue_length", "metrics.queue_a"),
        connect("junction-b.queue_length", "metrics.queue_b"),
        connect("junction-a.total_passed", "metrics.passed_a"),
        connect("junction-b.total_passed", "metrics.passed_b"),
    ]
    evt = AsyncioConnector.builder().build_event_connectors(comps)
    return LocalProcess(components=comps, connectors=conns + evt)

In [ ]:
OFFSET = CYCLE - 15  # ≈ mode travel time

proc_wave = build_process(light_b_offset=OFFSET)
async with proc_wave:
    await proc_wave.run()

mc_wave = proc_wave.components["metrics"]
df_queue_wave = pd.DataFrame(mc_wave.queue_history)
df_dep_wave = pd.DataFrame(mc_wave.departure_log)
df_cong_wave = pd.DataFrame(mc_wave.congestion_log)

print(f"Green-wave offset: {OFFSET}s")
print(f"Through Junction B: {df_queue_wave['passed_b'].iloc[-1]}  (was {passed_b})")
print(f"Congestion alerts : {len(df_cong_wave)}  (was {len(df_congestion)})")
if len(df_dep_wave):
    print(f"Mean wait (B)     : {df_dep_wave[df_dep_wave['from_junction']=='B']['wait_time'].mean():.1f}s")

### Side-by-side comparison

The top row shows the **synchronised** baseline; the bottom row shows the **green-wave** scenario.
Notice how Junction B's queue is dramatically reduced when its green phase is aligned with
arriving traffic from Junction A.

In [ ]:
fig = make_subplots(rows=2, cols=2, shared_xaxes=True, vertical_spacing=0.12, horizontal_spacing=0.08,
                    subplot_titles=("Baseline — Junction A", "Baseline — Junction B",
                                   "Green wave — Junction A", "Green wave — Junction B"))

# Baseline
fig.add_trace(go.Scatter(x=df_queue["time"], y=df_queue["queue_a"],
                         line=dict(color="royalblue"), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=df_queue["time"], y=df_queue["queue_b"],
                         line=dict(color="coral"), showlegend=False), row=1, col=2)

# Green wave
fig.add_trace(go.Scatter(x=df_queue_wave["time"], y=df_queue_wave["queue_a"],
                         line=dict(color="royalblue"), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=df_queue_wave["time"], y=df_queue_wave["queue_b"],
                         line=dict(color="coral"), showlegend=False), row=2, col=2)

for r in [1, 2]:
    for c in [1, 2]:
        fig.update_yaxes(title_text="Queue", row=r, col=c)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=2)

fig.update_layout(height=500, template="plotly_white",
                  title_text="Queue length comparison: synchronised vs green wave")
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_queue["time"], y=df_queue["passed_b"],
                         name="Baseline", line=dict(color="coral", dash="dash")))
fig.add_trace(go.Scatter(x=df_queue_wave["time"], y=df_queue_wave["passed_b"],
                         name="Green wave", line=dict(color="seagreen")))
fig.update_layout(template="plotly_white",
                  title="Cumulative vehicles through Junction B",
                  xaxis_title="Time (s)", yaxis_title="Vehicles exited")
fig.show()

## Summary

This demo illustrated several strengths of Plugboard's event-driven architecture applied to transport simulation:

1. **Hybrid communication** — regular connectors carry continuous state (time, signal phase, queue lengths)
   while discrete events (arrivals, departures, congestion alerts) propagate through the system.
2. **FSM logic** — traffic lights use a finite state machine that emits `SignalChangedEvent` only on transitions,
   avoiding unnecessary work when the state is unchanged.
3. **Stochastic processes** — vehicle arrivals follow a Poisson process and road travel times use a triangular
   distribution, giving the simulation realistic variability.
4. **Event monitoring** — the `MetricsCollector` subscribes to every event type, enabling rich post-simulation
   analysis without coupling the analytics to the simulation logic.
5. **What-if analysis** — by wrapping process creation in a helper function, we can easily sweep parameters
   (like signal offset) and compare scenarios.

Try experimenting with different arrival rates, signal timings, or road travel times to explore
how the system responds under stress.